# -----------------------------------------------------------
#       Make a copy of this notebook in your own Drive if you want to keep your results or changes!
# ----------------------------------------------------------

#


# Protein Backbone Generation with RFdiffusion

## Overview

This notebook is a hands-on demo of **RFdiffusion** (https://www.nature.com/articles/s41586-023-06415-8), a diffusion model that generates novel protein **backbone structures** — either completely unconditionally ("from noise"), or conditioned at inference time on a fixed structural motif (e.g. an enzyme's active site) that must be reproduced somewhere inside a new, otherwise entirely novel, protein scaffold.

**This is not fine-tuning.** RFdiffusion is used exactly as released — all of its structural knowledge comes from its own pretraining. We only use its built-in **inference-time conditioning**: a short `contigs` string that tells the model which residues (if any) to hold fixed in 3D space, and which residues to design completely from scratch around them. No training happens in this notebook.

Adapted from Sergey Ovchinnikov's [ColabDesign RFdiffusion notebook](https://github.com/sokrypton/ColabDesign/blob/main/rf/examples/diffusion.ipynb), which already handles weight loading, the SE(3)-Transformer build, and the 3D trajectory animation reused below.

---

## ⚠️ This notebook needs a GPU (unlike the CLEAN and BOES notebooks)

RFdiffusion's denoising loop is a real neural network forward pass at every one of ~50 diffusion steps — on CPU this is dramatically slower. **Before running anything below:**

Runtime → Change runtime type → Hardware accelerator → **GPU** (a free T4 is enough) → Save

With a GPU, each generation below should take roughly 1-3 minutes. On CPU, expect the same run to take much longer (potentially tens of minutes), and installation itself also takes a few minutes regardless of hardware.

---

## What is RFdiffusion?

RFdiffusion adapts **RoseTTAFold** (a protein structure *prediction* network) into a structure *generation* network, by training it to reverse a noising process applied to real protein backbones — similar in spirit to image diffusion models (e.g. Stable Diffusion), but operating on 3D coordinates and orientations instead of pixels. Starting from random noise, the model iteratively denoises a set of residue positions and orientations until they form a plausible protein backbone.

Because it is fundamentally a *conditional* generator, RFdiffusion can be steered at inference time in several ways without any retraining:
- **Unconditional generation** — no constraints, just "generate a plausible protein of this length."
- **Motif scaffolding** — fix a known functional motif (e.g. a binding site or catalytic residues) in place, and design a novel scaffold around it.
- **Binder design, symmetric oligomers, fold conditioning**, and more (not covered in this short demo — see the original ColabDesign notebook for those).

## Why is generating a protein backbone hard?

- **The design space is astronomically large.** Even a modest 100-residue backbone has a continuous 3D conformational space; only a tiny fraction of it folds into a stable, well-packed structure at all.
- **Local plausibility isn't enough.** A model has to get side-chain packing, secondary structure, and long-range tertiary contacts simultaneously right — a backbone that looks fine locally can still be globally unfoldable.
- **Function is even harder than fold.** A folded scaffold is necessary but not sufficient — for an *enzyme*, the catalytic residues also need to end up in the right relative 3D geometry, which is exactly what motif scaffolding targets directly.

## Why hen egg-white lysozyme as the motif-scaffolding example?

We deliberately picked an enzyme people are likely to have heard of, and one that is almost certainly extremely well-represented in RFdiffusion's (and the underlying RoseTTAFold's) training data: hen egg-white lysozyme was the **first enzyme structure ever solved by X-ray crystallography**, and has been used as a crystallography and biophysics test protein for decades — the PDB contains dozens of essentially this same structure. Its catalytic residues, **Glu35** and **Asp52**, are independently verified below against a real deposited PDB entry rather than taken on faith.

---

## Checkpoints used in this notebook

RFdiffusion ships several checkpoints fine-tuned for different tasks. This notebook uses two of them:

- **`Base_ckpt.pt`** — the general-purpose model, used below for **unconditional generation**.
- **`ActiveSite_ckpt.pt`** — fine-tuned specifically for scaffolding *very small* functional motifs such as enzyme active sites. Per RFdiffusion's own documentation: *"for scaffolding minimalist sites such as enzyme active sites, we fine-tuned RFdiffusion on examples similar to these tasks, allowing it to hold smaller motifs better in place."* Used below for the **lysozyme active-site scaffolding** part.

Both checkpoints are mirrored at [soldatmat/CZAI_Summer_School-RFdiffusion_weights](https://huggingface.co/datasets/soldatmat/CZAI_Summer_School-RFdiffusion_weights) on Hugging Face — a fast, reliable copy of the official checkpoints. The official host, `files.ipd.uw.edu` (a University of Washington file server), can be slow or briefly unreachable, which is exactly the kind of risk we want to avoid when many students download it at the same time during a live lecture; if the mirror is ever unavailable, the setup cell below falls back to the official host automatically.


> **Note on the checkpoint weights used below.** This notebook downloads `Base_ckpt.pt`
> and `ActiveSite_ckpt.pt` from our own Hugging Face mirror,
> [`soldatmat/CZAI_Summer_School-RFdiffusion_weights`](https://huggingface.co/datasets/soldatmat/CZAI_Summer_School-RFdiffusion_weights),
> rather than the official `files.ipd.uw.edu` host, which has been unreliable/unreachable.
> **These are not byte-identical to the official release** -- they come from a third-party
> mirror and their MD5 checksums differ from the official ones (both are documented, and
> checked, in the download cell below). We're using them anyway because RFdiffusion's BSD
> license explicitly permits redistributing the weights, and because we independently
> verified these specific files are functionally correct: they load `strict=True` into the
> real model architecture with no NaNs, and a CPU test run using them to hold the lysozyme
> Glu35/Asp52 active site in place reproduced that geometry to within a fraction of an
> angstrom (see the Discussion at the end of this notebook). Full provenance and checksums
> are in the mirror's README.

In [1]:
#@title Setup: install RFdiffusion, ColabDesign, and download weights (~3-6 min) { display-mode: "form" }
import os, sys, json, hashlib, subprocess, time

_setup_start = time.time()  # %%time can only be a cell's first line, which would collide
                            # with the #@title Colab-form comment above, so we time this
                            # cell manually instead.

RFDIFFUSION_VENV = os.path.abspath("rfdiff_venv")
RFDIFFUSION_VENV_PYTHON = os.path.join(RFDIFFUSION_VENV, "bin", "python3")
RFDIFFUSION_VENV_READY = os.path.join(RFDIFFUSION_VENV, ".ready")  # written only once every install below succeeds

if not os.path.isdir("RFdiffusion"):
    print("installing RFdiffusion...")
    os.system("git clone https://github.com/sokrypton/RFdiffusion.git")

if not os.path.isdir("colabdesign"):
    print("installing ColabDesign...")
    os.system("pip -q install git+https://github.com/sokrypton/ColabDesign.git")
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")


def _set_rfdiffusion_ld_library_path():
    """So the Python 3.11 environment's cu124-built dgl can dlopen its CUDA
    runtime libraries even on a CPU-only Colab instance with no system CUDA
    install of its own -- a real GPU runtime already provides these, but this
    notebook explicitly documents a CPU fallback (just slower) as supported, so
    students who forgot to switch runtime type or ran out of GPU quota still get
    a working import instead of a dlopen failure."""
    venv_site = subprocess.run(
        [RFDIFFUSION_VENV_PYTHON, "-c", "import site; print(site.getsitepackages()[0])"],
        capture_output=True, text=True,
    ).stdout.strip()
    cuda_lib_dirs = ":".join(
        f"{venv_site}/nvidia/{pkg}/lib"
        for pkg in ("cuda_runtime", "cublas", "cusparse", "curand", "cusolver")
    )
    os.environ["LD_LIBRARY_PATH"] = f"{cuda_lib_dirs}:{os.environ.get('LD_LIBRARY_PATH', '')}"


if not os.path.isfile(RFDIFFUSION_VENV_READY):
    # WHY RFdiffusion inference runs in a second, older Python interpreter instead
    # of this notebook's own kernel: `dgl` -- the graph library RFdiffusion's
    # SE3Transformer depends on -- has never published a wheel for Python >= 3.13,
    # not on PyPI and not on its own custom wheel index (data.dgl.ai), and it has
    # never shipped a source distribution either. On a Python this recent,
    # `pip install dgl` silently falls back to `dgl 0.1.3` (2018!), which is
    # missing APIs RFdiffusion actually needs (`dgl.graph()`,
    # `dgl.nn.pytorch.AvgPooling`, etc.) -- a hard blocker that pinning alone
    # cannot fix within this process's own Python version. So instead we build a
    # real Python 3.11 environment (which DOES have functional dgl wheels) in a
    # venv alongside this notebook, install RFdiffusion's own runtime deps + dgl +
    # SE3Transformer into THAT environment, and talk to it over subprocess calls +
    # files rather than importing it directly. Everything below this block
    # (fix_contigs, fix_pdb, the 3D trajectory animation) runs fine in this
    # notebook's own Python because none of it touches dgl -- only RFdiffusion's
    # own inference code does, and that now lives entirely in the Python 3.11
    # environment.
    print(
        f"this notebook's Python ({sys.version.split()[0]}) has no working dgl -- "
        "creating a Python 3.11 environment for RFdiffusion inference..."
    )
    os.system("pip install -q uv")
    _UV = f"{sys.executable} -m uv"
    _uv_py_ok = os.system(f"{_UV} python install 3.11") == 0
    if not _uv_py_ok:
        raise RuntimeError(
            "could not install a Python 3.11 interpreter via uv (see the printed "
            "output above). RFdiffusion inference needs a Python with a working "
            "dgl wheel -- currently Python 3.11 -- and this notebook has no other "
            "way to obtain one."
        )
    os.system(f"{_UV} venv --python 3.11 {RFDIFFUSION_VENV}")
    # Plain `pip install torch` (no special index): modern PyPI torch wheels are
    # CUDA-enabled by default on Linux, so this environment gets real GPU support
    # on Colab -- same as this notebook's own kernel already relies on Colab's
    # preinstalled torch being CUDA-enabled.
    os.system(f"{_UV} pip install --python {RFDIFFUSION_VENV_PYTHON} -q torch")
    os.system(
        f"{_UV} pip install --python {RFDIFFUSION_VENV_PYTHON} -q "
        "numpy scipy jedi omegaconf hydra-core icecream pyrsistent pynvml decorator "
        # requests/tqdm/psutil/pandas/networkx: not RFdiffusion imports directly, but
        # dgl's own submodules (e.g. dgl.dataloading) import psutil unconditionally,
        # and other dgl codepaths pull in the rest -- this notebook's own kernel gets
        # all of these for free from Colab's preinstalled packages, but this fresh
        # venv needs them installed explicitly.
        "opt_einsum requests tqdm psutil pandas networkx"
    )
    os.system(
        f"{_UV} pip install --python {RFDIFFUSION_VENV_PYTHON} -q "
        "git+https://github.com/NVIDIA/dllogger#egg=dllogger"
    )
    os.system(
        f"{_UV} pip install --python {RFDIFFUSION_VENV_PYTHON} -q "
        "nvidia-cuda-runtime-cu12==12.4.127 nvidia-cublas-cu12 nvidia-cusparse-cu12 "
        "nvidia-curand-cu12 nvidia-cusolver-cu12"
    )

    # dgl: pin to a specific, known-good version instead of leaving it unconstrained,
    # for exactly the same reason explained above -- an unconstrained `pip install
    # dgl` on a Python without a real wheel silently resolves to the ancient 0.1.3.
    # Pinning surfaces that mismatch immediately and clearly (see the functional
    # check below) instead of failing much later with a confusing traceback.
    _DGL_PIN = "dgl==2.4.0+cu124"
    _DGL_INDEX = "https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html"
    _dgl_pin_ok = os.system(
        f"{_UV} pip install --python {RFDIFFUSION_VENV_PYTHON} -q --no-deps {_DGL_PIN} -f {_DGL_INDEX}"
    ) == 0
    if not _dgl_pin_ok:
        raise RuntimeError(
            f"could not install the pinned {_DGL_PIN} into the Python 3.11 "
            f"environment -- check https://data.dgl.ai/wheels/ for currently "
            "available builds and update this pin."
        )
    os.system(
        f"{_UV} pip install --python {RFDIFFUSION_VENV_PYTHON} -q --no-deps e3nn==0.5.5 opt_einsum_fx"
    )
    _se3_ok = os.system(
        f"cd RFdiffusion/env/SE3Transformer && {_UV} pip install --python {RFDIFFUSION_VENV_PYTHON} -q ."
    ) == 0
    if not _se3_ok:
        raise RuntimeError(
            "failed to install RFdiffusion's SE3Transformer package into the "
            "Python 3.11 environment."
        )

    _set_rfdiffusion_ld_library_path()

    # Fail loudly and clearly here if the installed dgl is functional but too old
    # for RFdiffusion's SE3Transformer, rather than several subprocess frames deep
    # during the first real generation run.
    _dgl_check = subprocess.run(
        [
            RFDIFFUSION_VENV_PYTHON, "-c",
            "import dgl, sys\n"
            "missing = []\n"
            "if not callable(getattr(dgl, 'graph', None)): missing.append('graph')\n"
            "if not hasattr(dgl, 'ops'): missing.append('ops')\n"
            "try:\n"
            "    from dgl.nn.pytorch import AvgPooling\n"
            "except ImportError:\n"
            "    missing.append('nn.pytorch.AvgPooling')\n"
            "if missing:\n"
            "    print('MISSING:', missing); sys.exit(1)\n"
            "print(f'dgl {dgl.__version__} OK')\n",
        ],
        capture_output=True, text=True,
    )
    print(_dgl_check.stdout.strip())
    if _dgl_check.returncode != 0:
        raise RuntimeError(
            f"Installed dgl in the Python 3.11 environment is missing APIs "
            f"RFdiffusion's SE3Transformer needs: "
            f"{_dgl_check.stderr.strip() or _dgl_check.stdout.strip()}"
        )

    with open(RFDIFFUSION_VENV_READY, "w") as _f:
        _f.write("ok\n")
else:
    _set_rfdiffusion_ld_library_path()

# Two small scripts that run *inside* the Python 3.11 environment above -- this
# notebook's own Python 3.13 kernel never imports anything from RFdiffusion's own
# package (that's what triggers the dgl import chain), it only shells out to these.
_RESOLVE_CONTIGS_SCRIPT = "rfdiffusion_resolve_contigs.py"
with open(_RESOLVE_CONTIGS_SCRIPT, "w") as f:
    f.write('''"""Run inside the Python 3.11 RFdiffusion environment: parse a PDB file with
RFdiffusion's own `inference.utils.parse_pdb` (needed so `fix_contigs` below can
resolve chain/residue-range motif syntax like "A33-37" against it) and print back
just the (chain, residue number) index, as JSON, for the notebook's own Python to
use. Importing `inference.utils` pulls in RFdiffusion's full model stack (and
therefore dgl) as a side effect of its own internal imports, which is exactly why
this has to run here instead of in the notebook's Python 3.13 kernel.
"""
import sys, json

sys.path.append(sys.argv[1])  # path to the cloned RFdiffusion repo
from inference.utils import parse_pdb

parsed = parse_pdb(sys.argv[2])
print(json.dumps({"pdb_idx": [[c, int(i)] for c, i in parsed["pdb_idx"]]}))
''')

_RUN_INFERENCE_SCRIPT = "rfdiffusion_run_inference.py"
with open(_RUN_INFERENCE_SCRIPT, "w") as f:
    f.write('''"""Run inside the Python 3.11 RFdiffusion environment: launch RFdiffusion's own
run_inference.py exactly as it ships, with the same Hydra config overrides the
notebook already built. Usage: rfdiffusion_run_inference.py <RFdiffusion dir> <hydra overrides...>
"""
import contextlib
import os
import runpy
import sys

rfdiffusion_dir = sys.argv[1]
hydra_opts = sys.argv[2:]
sys.path.insert(0, rfdiffusion_dir)

import torch

if not torch.cuda.is_available():
    # SE3Transformer's layers call torch.cuda.nvtx.range() unconditionally on
    # every forward pass; that's a harmless marker on a real GPU, but raises with
    # no CUDA context at all -- make it a no-op so CPU-only fallback still works.
    @contextlib.contextmanager
    def _noop_range(*a, **kw):
        yield

    torch.cuda.nvtx.range = _noop_range
    torch.cuda.nvtx.range_push = lambda *a, **kw: None
    torch.cuda.nvtx.range_pop = lambda *a, **kw: None

run_inference_path = os.path.join(rfdiffusion_dir, "run_inference.py")
sys.argv = [run_inference_path] + hydra_opts
runpy.run_path(run_inference_path, run_name="__main__")
''')

if not os.path.isdir("RFdiffusion/models"):
    print("downloading RFdiffusion weights...")
    os.makedirs("RFdiffusion/models", exist_ok=True)

    # Expected MD5, keyed by WHERE the file came from -- not a single hardcoded
    # hash. Our HF mirror currently hosts an alternate-source copy of these
    # checkpoints whose bytes differ from the official files.ipd.uw.edu release
    # (see the markdown note above and the dataset's README for why), so the two
    # sources have two different correct checksums. If the official host ever
    # comes back and gets used as the fallback, its download must be checked
    # against the OFFICIAL hash, not the mirror's -- otherwise a perfectly good
    # official download would look like a checksum failure.
    OFFICIAL_MD5 = {
        "Base_ckpt.pt": "6f5902ac237024bdd0c176cb93063dc4",
        "ActiveSite_ckpt.pt": "5532d2e1f3a4738decd58b19d633b3c3",
    }
    HF_MIRROR_MD5 = {
        "Base_ckpt.pt": "4aa4a27ba280d23541e01860c106c7cc",
        "ActiveSite_ckpt.pt": "0d9f82af03c73011c6fec060bac5b731",
    }
    EXPECTED_MD5_BY_SOURCE = {
        "hf_mirror": HF_MIRROR_MD5,
        "official": OFFICIAL_MD5,
    }
    OFFICIAL_URL = {
        name: f"http://files.ipd.uw.edu/pub/RFdiffusion/{md5}/{name}"
        for name, md5 in OFFICIAL_MD5.items()
    }
    HF_WEIGHTS_REPO = "soldatmat/CZAI_Summer_School-RFdiffusion_weights"

    def md5sum(path):
        h = hashlib.md5()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        return h.hexdigest()

    from huggingface_hub import hf_hub_download

    for ckpt in OFFICIAL_MD5:
        dest = f"RFdiffusion/models/{ckpt}"
        try:
            cached = hf_hub_download(repo_id=HF_WEIGHTS_REPO, repo_type="dataset", filename=ckpt)
            os.system(f"cp {cached} {dest}")
            source = "hf_mirror"
            print(f"  {ckpt}: downloaded from the HF mirror")
        except Exception as e:
            print(f"  {ckpt}: HF mirror unavailable ({e}); falling back to files.ipd.uw.edu ...")
            os.system(f"wget -q -O {dest} {OFFICIAL_URL[ckpt]}")
            source = "official"

        expected_md5 = EXPECTED_MD5_BY_SOURCE[source][ckpt]
        got_md5 = md5sum(dest)
        ok = "OK" if got_md5 == expected_md5 else "MISMATCH -- re-download recommended"
        print(f"  {ckpt}: source={source} md5={got_md5} (expected {expected_md5} for this source) [{ok}]")

    # Cached IGSO(3) noise schedules speed up the first diffusion run; this is
    # a best-effort optimization only -- if it's unavailable, RFdiffusion just
    # computes the schedules itself (a bit slower) the first time it's used.
    os.system(
        "wget -q https://files.ipd.uw.edu/krypton/schedules.zip "
        "&& unzip -q -o schedules.zip -d RFdiffusion && rm -f schedules.zip"
    )

import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import ipywidgets as widgets
import py3Dmol
from string import ascii_uppercase, ascii_lowercase

from colabdesign.rf.utils import get_ca, get_Ls, fix_contigs, fix_pdb, make_animation
from colabdesign.shared.protein import pdb_to_string
from colabdesign.shared.plot import plot_pseudo_3D, pymol_color_list

alphabet_list = list(ascii_uppercase + ascii_lowercase)

print("\nPyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> GPU, then re-run this cell.")
print(f"\nSetup took {time.time() - _setup_start:.1f}s")

installing RFdiffusion...


Cloning into 'RFdiffusion'...


installing ColabDesign...


downloading RFdiffusion weights...


  Base_ckpt.pt: downloaded from the HF mirror


  Base_ckpt.pt: source=hf_mirror md5=4aa4a27ba280d23541e01860c106c7cc (expected 4aa4a27ba280d23541e01860c106c7cc for this source) [OK]


  ActiveSite_ckpt.pt: downloaded from the HF mirror


  ActiveSite_ckpt.pt: source=hf_mirror md5=0d9f82af03c73011c6fec060bac5b731 (expected 0d9f82af03c73011c6fec060bac5b731 for this source) [OK]



PyTorch version: 2.14.0
CUDA available: False

Setup took 21.1s


In [2]:
#@title Helper functions for running RFdiffusion (adapted from ColabDesign) { display-mode: "form" }
import time, random, string, signal, json, subprocess

# macOS has no /dev/shm (that's a Linux-only tmpfs mount, not creatable
# from userland); use a local scratch directory instead as the shared
# hand-off point between the RFdiffusion subprocess (writer) and this
# notebook's own polling loop below (reader) -- same role, different path.
DEV_SHM_PATH = os.path.abspath('./dev_shm')
os.makedirs(DEV_SHM_PATH, exist_ok=True)

def get_pdb(pdb_code):
    """Fetch a 4-letter PDB code's biological assembly (used for the lysozyme motif below)."""
    if os.path.isfile(pdb_code):
        return pdb_code
    if not os.path.isfile(f"{pdb_code}.pdb1"):
        os.system(f"wget -qnc https://files.rcsb.org/download/{pdb_code}.pdb1.gz")
        os.system(f"gunzip -f {pdb_code}.pdb1.gz")
    return f"{pdb_code}.pdb1"


def run(command, steps, num_designs=1, visual="none"):
    """Launch run_inference.py (in the Python 3.11 RFdiffusion environment) and show
    a live progress bar (+ optional live preview) by watching the per-step PDB dumps
    RFdiffusion writes to DEV_SHM_PATH (a local scratch dir standing in for /dev/shm on macOS) -- a real filesystem path shared by every process
    on this VM regardless of which Python wrote to it, so this works unchanged
    whether `command` runs in this kernel's Python or a different one."""

    def run_command_and_get_pid(command):
        pid_file = f"{DEV_SHM_PATH}/pid"
        os.system(f"nohup {command} & echo $! > {pid_file}")
        with open(pid_file, "r") as f:
            pid = int(f.read().strip())
        os.remove(pid_file)
        return pid

    def is_process_running(pid):
        try:
            os.kill(pid, 0)
        except OSError:
            return False
        return True

    run_output = widgets.Output()
    progress = widgets.FloatProgress(min=0, max=1, description="running", bar_style="info")
    display(widgets.VBox([progress, run_output]))

    for n in range(steps):
        if os.path.isfile(f"{DEV_SHM_PATH}/{n}.pdb"):
            os.remove(f"{DEV_SHM_PATH}/{n}.pdb")

    pid = run_command_and_get_pid(command)
    try:
        fail = False
        for _ in range(num_designs):
            for n in range(steps):
                wait = True
                while wait and not fail:
                    time.sleep(0.1)
                    if os.path.isfile(f"{DEV_SHM_PATH}/{n}.pdb"):
                        pdb_str = open(f"{DEV_SHM_PATH}/{n}.pdb").read()
                        if pdb_str[-3:] == "TER":
                            wait = False
                        elif not is_process_running(pid):
                            fail = True
                    elif not is_process_running(pid):
                        fail = True
                if fail:
                    progress.bar_style = "danger"
                    progress.description = "failed"
                    break
                progress.value = (n + 1) / steps
                if visual != "none":
                    with run_output:
                        run_output.clear_output(wait=True)
                        if visual == "image":
                            xyz, bfact = get_ca(f"{DEV_SHM_PATH}/{n}.pdb", get_bfact=True)
                            fig = plt.figure()
                            fig.set_dpi(100); fig.set_figwidth(6); fig.set_figheight(6)
                            ax1 = fig.add_subplot(111); ax1.set_xticks([]); ax1.set_yticks([])
                            plot_pseudo_3D(xyz, c=bfact, cmin=0.5, cmax=0.9, ax=ax1)
                            plt.show()
                        elif visual == "interactive":
                            view = py3Dmol.view(js="https://3dmol.org/build/3Dmol.js")
                            view.addModel(pdb_str, "pdb")
                            view.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 0.5, "max": 0.9}}})
                            view.zoomTo()
                            view.show()
                if os.path.exists(f"{DEV_SHM_PATH}/{n}.pdb"):
                    os.remove(f"{DEV_SHM_PATH}/{n}.pdb")
            if fail:
                progress.bar_style = "danger"
                progress.description = "failed"
                break
        while is_process_running(pid):
            time.sleep(0.1)
    except KeyboardInterrupt:
        os.kill(pid, signal.SIGTERM)
        progress.bar_style = "danger"
        progress.description = "stopped"


def run_diffusion(name, contigs, pdb=None, iterations=50, num_designs=1,
                   ckpt_override=None, visual="none"):
    """Run RFdiffusion. `contigs` follows ColabDesign's mini-language:
         '100'                              -> free 100-residue monomer (unconditional)
         '30-40/A33-37/8-14/A50-54/30-40'   -> two fixed motif windows taken from `pdb`,
                                                with new residues freely diffused around
                                                and between them (motif scaffolding)
    Returns (path, resolved_contigs) for use by show_trajectory().

    The actual inference (`run_inference.py`) and any RFdiffusion-internal PDB
    parsing both run inside the Python 3.11 environment set up in the setup cell
    above, via subprocess -- this notebook's own Python never imports anything
    from RFdiffusion's own package, only from ColabDesign (which doesn't touch dgl).
    """
    path = name
    while os.path.exists(f"outputs/{path}_0.pdb"):
        path = name + "_" + "".join(random.choices(string.ascii_lowercase + string.digits, k=5))
    full_path = f"outputs/{path}"
    os.makedirs(full_path, exist_ok=True)

    opts = [f"inference.output_prefix={full_path}", f"inference.num_designs={num_designs}"]

    contig_list = contigs.replace(",", " ").split()
    is_fixed = any(seg.split("-")[0][:1].isalpha() for c in contig_list for seg in c.split("/") if seg)

    if is_fixed:
        assert pdb is not None, "contigs reference a chain (e.g. 'A33-37') but no `pdb` was given"
        pdb_str = pdb_to_string(get_pdb(pdb))
        pdb_filename = f"{full_path}/input.pdb"
        with open(pdb_filename, "w") as handle:
            handle.write(pdb_str)

        # RFdiffusion's own `inference.utils.parse_pdb` pulls in its full model
        # stack (and therefore dgl) as a side effect of its own imports, so it has
        # to run inside the Python 3.11 environment -- see the setup cell.
        resolve = subprocess.run(
            [RFDIFFUSION_VENV_PYTHON, "rfdiffusion_resolve_contigs.py", "RFdiffusion", pdb_filename],
            capture_output=True, text=True,
        )
        if resolve.returncode != 0:
            raise RuntimeError(f"failed to parse {pdb_filename} in the RFdiffusion environment:\n{resolve.stderr}")
        parsed_pdb = {"pdb_idx": [tuple(entry) for entry in json.loads(resolve.stdout)["pdb_idx"]]}

        opts.append(f"inference.input_pdb={pdb_filename}")
        contig_list = fix_contigs(contig_list, parsed_pdb)
    else:
        contig_list = fix_contigs(contig_list, None)

    opts.append(f"diffuser.T={iterations}")
    opts.append(f"'contigmap.contigs=[{' '.join(contig_list)}]'")
    opts += ["inference.dump_pdb=True", "inference.dump_pdb_path='" + DEV_SHM_PATH + "'"]
    if ckpt_override:
        opts.append(f"inference.ckpt_override_path={ckpt_override}")

    print("output:", full_path)
    print("contigs:", contig_list)
    cmd = f"{RFDIFFUSION_VENV_PYTHON} rfdiffusion_run_inference.py RFdiffusion {' '.join(opts)}"
    print(cmd)
    run(cmd, iterations, num_designs, visual=visual)

    for n in range(num_designs):
        for pdb_file in [f"outputs/traj/{path}_{n}_pX0_traj.pdb",
                          f"outputs/traj/{path}_{n}_Xt-1_traj.pdb",
                          f"{full_path}_{n}.pdb"]:
            with open(pdb_file) as handle:
                pdb_txt = handle.read()
            with open(pdb_file, "w") as handle:
                handle.write(fix_pdb(pdb_txt, contig_list))

    return path, contig_list


def show_trajectory(path, contigs, animate="movie", color="chain", dpi=100, denoise=True, num_design=0):
    """3D visualization of the denoising trajectory -- adapted directly from the
    'Display 3D structure' cell in ColabDesign's diffusion.ipynb."""
    pdb_traj = f"outputs/traj/{path}_{num_design}_{'pX0' if denoise else 'Xt-1'}_traj.pdb"

    if animate in ["none", "interactive"]:
        view = py3Dmol.view(js="https://3dmol.org/build/3Dmol.js")
        if animate == "interactive":
            pdb_str = open(pdb_traj, "r").read()
            view.addModelsAsFrames(pdb_str, "pdb", {"hbondCutoff": 4.0})
        else:
            pdb_str = open(f"outputs/{path}_{num_design}.pdb", "r").read()
            view.addModel(pdb_str, "pdb", {"hbondCutoff": 4.0})
        if color == "rainbow":
            view.setStyle({"cartoon": {"color": "spectrum"}})
        elif color == "chain":
            for n, chain, c in zip(range(len(contigs)), alphabet_list, pymol_color_list):
                view.setStyle({"chain": chain}, {"cartoon": {"color": c}})
        else:
            view.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 0.5, "max": 0.9}}})
        view.zoomTo()
        if animate == "interactive":
            view.animate({"loop": "backAndForth"})
        view.show()
    else:  # "movie"
        Ls = get_Ls(contigs)
        xyz, bfact = get_ca(pdb_traj, get_bfact=True)
        xyz = xyz.reshape((-1, sum(Ls), 3))[::-1]
        bfact = bfact.reshape((-1, sum(Ls)))[::-1]
        if color == "chain":
            display(HTML(make_animation(xyz, Ls=Ls, dpi=dpi, ref=-1)))
        elif color == "rainbow":
            display(HTML(make_animation(xyz, dpi=dpi, ref=-1)))
        else:
            display(HTML(make_animation(xyz, plddt=bfact * 100, dpi=dpi, ref=-1)))

---

## Part 1 — Unconditional generation: watch a protein emerge from noise

No constraints here: we ask RFdiffusion for a single ~100-residue monomer and let it design a completely novel fold. Internally, the model starts from random noise for every residue's position and orientation and iteratively denoises it over `iterations` steps; at each step it also produces a full prediction of the final structure (called `pX0`), and it is exactly this sequence of `pX0` predictions across all steps that we animate below — this is the "protein folding out of noise" visual.


In [3]:
#@title Run RFdiffusion: unconditional backbone generation
name = "unconditional_demo"  #@param {type:"string"}
contigs = "100"  #@param {type:"string"}
iterations = 50  #@param [25, 50, 100, 150, 200] {type:"raw"}
num_designs = 1  #@param [1, 2, 4] {type:"raw"}

path_uncond, contigs_uncond = run_diffusion(
    name=name, contigs=contigs, pdb=None, iterations=iterations,
    num_designs=num_designs, ckpt_override=None, visual="none",
)


output: outputs/unconditional_demo
contigs: ['100-100']
/private/tmp/claude-501/-Users-soldatmat-Documents-events-CZAI-summer-school-2026/caf8c02d-0b75-4992-a3b6-c2fd1f6a15d6/scratchpad/rfdiffusion_execution/rfdiff_venv/bin/python3 rfdiffusion_run_inference.py RFdiffusion inference.output_prefix=outputs/unconditional_demo inference.num_designs=1 diffuser.T=50 'contigmap.contigs=[100-100]' inference.dump_pdb=True inference.dump_pdb_path='/private/tmp/claude-501/-Users-soldatmat-Documents-events-CZAI-summer-school-2026/caf8c02d-0b75-4992-a3b6-c2fd1f6a15d6/scratchpad/rfdiffusion_execution/dev_shm'


/private/tmp/claude-501/-Users-soldatmat-Documents-events-CZAI-summer-school-2026/caf8c02d-0b75-4992-a3b6-c2fd1f6a15d6/scratchpad/rfdiffusion_execution/rfdiff_venv/lib/python3.12/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


RFdiffusion/run_inference.py:55: SyntaxWarning: invalid escape sequence '\d'
  m = re.match(".*_(\d+)\.pdb$", e)


[2026-09-10 00:05:10,660][inference.model_runners][INFO] - Reading checkpoint from /private/tmp/claude-501/-Users-soldatmat-Documents-events-CZAI-summer-school-2026/caf8c02d-0b75-4992-a3b6-c2fd1f6a15d6/scratchpad/rfdiffusion_execution/RFdiffusion/inference/../models/Base_ckpt.pt


This is inf_conf.ckpt_path
/private/tmp/claude-501/-Users-soldatmat-Documents-events-CZAI-summer-school-2026/caf8c02d-0b75-4992-a3b6-c2fd1f6a15d6/scratchpad/rfdiffusion_execution/RFdiffusion/inference/../models/Base_ckpt.pt
Assembling -model, -diffuser and -preprocess configs from checkpoint
USING MODEL CONFIG: self._conf[model][n_extra_block] = 4
USING MODEL CONFIG: self._conf[model][n_main_block] = 32
USING MODEL CONFIG: self._conf[model][n_ref_block] = 4
USING MODEL CONFIG: self._conf[model][d_msa] = 256
USING MODEL CONFIG: self._conf[model][d_msa_full] = 64
USING MODEL CONFIG: self._conf[model][d_pair] = 128
USING MODEL CONFIG: self._conf[model][d_templ] = 64
USING MODEL CONFIG: self._conf[model][n_head_msa] = 8
USING MODEL CONFIG: self._conf[model][n_head_pair] = 4
USING MODEL CONFIG: self._conf[model][n_head_templ] = 4
USING MODEL CONFIG: self._conf[model][d_hidden] = 32
USING MODEL CONFIG: self._conf[model][d_hidden_templ] = 32
USING MODEL CONFIG: self._conf[model][p_drop] = 0.1

...

[2026-09-10 00:07:44,338][inference.model_runners][INFO] - Timestep 5, input to next step: ----------------------------------------------------------------------------------------------------


[2026-09-10 00:07:47,199][inference.model_runners][INFO] - Timestep 4, input to next step: ----------------------------------------------------------------------------------------------------


[2026-09-10 00:07:50,035][inference.model_runners][INFO] - Timestep 3, input to next step: ----------------------------------------------------------------------------------------------------


[2026-09-10 00:07:52,910][inference.model_runners][INFO] - Timestep 2, input to next step: ----------------------------------------------------------------------------------------------------


[2026-09-10 00:07:56,102][__main__][INFO] - Finished design in 2.66 minutes


In [4]:
#@title Show the denoising trajectory in 3D {run: "auto"}
animate = "movie"  #@param ["movie", "interactive", "none"]
color = "chain"    #@param ["rainbow", "chain", "plddt"]
dpi = 100          #@param [100, 200, 400] {type:"raw"}

show_trajectory(path_uncond, contigs_uncond, animate=animate, color=color, dpi=dpi)


---

## Part 2 — Motif scaffolding: designing a new enzyme scaffold around lysozyme's real active site

### Verifying the active site ourselves

Rather than trusting the commonly-cited "Glu35 / Asp52" description of hen egg-white lysozyme's catalytic residues, we check it against a real deposited structure: **[PDB 1LYZ](https://www.rcsb.org/structure/1LYZ)** (Diamond, 1974; 2.0 Å, chain A, 129/129 residues present, no gaps — a clean, complete, single-chain entry). Downloading and parsing that file confirms:

- **Residue 35, chain A → `GLU`** (Glu35) ✓
- **Residue 52, chain A → `ASP`** (Asp52) ✓

matching the textbook description exactly, in this specific numbering, in this specific deposited structure.

### Building the `contigs` motif-scaffolding string

Glu35 and Asp52 are **17 residues apart in sequence** but close together in 3D space (this is exactly why they can jointly act as the catalytic pair). To scaffold them, we fix two small windows of chain A — `A33-37` (Lys33–Phe34–**Glu35**–Ser36–Asn37) and `A50-54` (Ser50–Thr51–**Asp52**–Tyr53–Gly54) — each just a modest few residues around its catalytic residue, per standard motif-scaffolding practice (RFdiffusion's own docs specifically recommend the `ActiveSite_ckpt.pt` checkpoint for holding *small* motifs like this one in place). Everything else — the N-terminus, the 12-residue loop that connects the two windows in the native protein, and the C-terminus — is freely diffused as new residues, so RFdiffusion has to invent an entirely new scaffold that nonetheless holds both catalytic side chains in their correct relative geometry:

```
contigs = "30-40/A33-37/8-14/A50-54/30-40"
```

i.e.: 30-40 new residues, then fixed `A33-37`, then an 8-14-residue new loop (replacing the native 12-residue linker with something RFdiffusion designs itself), then fixed `A50-54`, then 30-40 more new residues — a novel ~80-105 residue protein containing a real catalytic dyad.


In [5]:
#@title Run RFdiffusion: scaffold a new protein around lysozyme's active site
name = "lysozyme_activesite_demo"  #@param {type:"string"}
pdb_code = "1LYZ"  #@param {type:"string"}
contigs = "30-40/A33-37/8-14/A50-54/30-40"  #@param {type:"string"}
iterations = 50  #@param [25, 50, 100, 150, 200] {type:"raw"}
num_designs = 1  #@param [1, 2, 4] {type:"raw"}

path_motif, contigs_motif = run_diffusion(
    name=name, contigs=contigs, pdb=pdb_code, iterations=iterations,
    num_designs=num_designs,
    ckpt_override="./RFdiffusion/models/ActiveSite_ckpt.pt",
    visual="none",
)


output: outputs/lysozyme_activesite_demo
contigs: ['33-33/A33-37/11-11/A50-54/30-30']
/private/tmp/claude-501/-Users-soldatmat-Documents-events-CZAI-summer-school-2026/caf8c02d-0b75-4992-a3b6-c2fd1f6a15d6/scratchpad/rfdiffusion_execution/rfdiff_venv/bin/python3 rfdiffusion_run_inference.py RFdiffusion inference.output_prefix=outputs/lysozyme_activesite_demo inference.num_designs=1 inference.input_pdb=outputs/lysozyme_activesite_demo/input.pdb diffuser.T=50 'contigmap.contigs=[33-33/A33-37/11-11/A50-54/30-30]' inference.dump_pdb=True inference.dump_pdb_path='/private/tmp/claude-501/-Users-soldatmat-Documents-events-CZAI-summer-school-2026/caf8c02d-0b75-4992-a3b6-c2fd1f6a15d6/scratchpad/rfdiffusion_execution/dev_shm' inference.ckpt_override_path=./RFdiffusion/models/ActiveSite_ckpt.pt


/private/tmp/claude-501/-Users-soldatmat-Documents-events-CZAI-summer-school-2026/caf8c02d-0b75-4992-a3b6-c2fd1f6a15d6/scratchpad/rfdiffusion_execution/rfdiff_venv/lib/python3.12/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


RFdiffusion/run_inference.py:55: SyntaxWarning: invalid escape sequence '\d'
  m = re.match(".*_(\d+)\.pdb$", e)


[2026-09-10 00:08:01,215][inference.model_runners][INFO] - Reading checkpoint from ./RFdiffusion/models/ActiveSite_ckpt.pt


This is inf_conf.ckpt_path
./RFdiffusion/models/ActiveSite_ckpt.pt
Assembling -model, -diffuser and -preprocess configs from checkpoint
USING MODEL CONFIG: self._conf[model][n_extra_block] = 4
USING MODEL CONFIG: self._conf[model][n_main_block] = 32
USING MODEL CONFIG: self._conf[model][n_ref_block] = 4
USING MODEL CONFIG: self._conf[model][d_msa] = 256
USING MODEL CONFIG: self._conf[model][d_msa_full] = 64
USING MODEL CONFIG: self._conf[model][d_pair] = 128
USING MODEL CONFIG: self._conf[model][d_templ] = 64
USING MODEL CONFIG: self._conf[model][n_head_msa] = 8
USING MODEL CONFIG: self._conf[model][n_head_pair] = 4
USING MODEL CONFIG: self._conf[model][n_head_templ] = 4
USING MODEL CONFIG: self._conf[model][d_hidden] = 32
USING MODEL CONFIG: self._conf[model][d_hidden_templ] = 32
USING MODEL CONFIG: self._conf[model][p_drop] = 0.15
USING MODEL CONFIG: self._conf[model][SE3_param_full] = {'num_layers': 1, 'num_channels': 32, 'num_degrees': 2, 'n_heads': 4, 'div': 4, 'l0_in_features': 8

...

[2026-09-10 00:09:38,337][inference.utils][INFO] - Sampled motif RMSD: 0.30
[2026-09-10 00:09:38,338][inference.model_runners][INFO] - Timestep 5, input to next step: ---------------------------------KFESN-----------STDYG------------------------------


[2026-09-10 00:09:40,500][inference.utils][INFO] - Sampled motif RMSD: 0.29
[2026-09-10 00:09:40,500][inference.model_runners][INFO] - Timestep 4, input to next step: ---------------------------------KFESN-----------STDYG------------------------------


[2026-09-10 00:09:42,627][inference.utils][INFO] - Sampled motif RMSD: 0.30
[2026-09-10 00:09:42,627][inference.model_runners][INFO] - Timestep 3, input to next step: ---------------------------------KFESN-----------STDYG------------------------------


[2026-09-10 00:09:44,737][inference.utils][INFO] - Sampled motif RMSD: 0.29
[2026-09-10 00:09:44,738][inference.model_runners][INFO] - Timestep 2, input to next step: ---------------------------------KFESN-----------STDYG------------------------------


[2026-09-10 00:09:47,169][__main__][INFO] - Finished design in 1.75 minutes


In [6]:
#@title Show the active-site scaffolding trajectory in 3D {run: "auto"}
animate = "movie"  #@param ["movie", "interactive", "none"]
color = "chain"    #@param ["rainbow", "chain", "plddt"]
dpi = 100          #@param [100, 200, 400] {type:"raw"}

show_trajectory(path_motif, contigs_motif, animate=animate, color=color, dpi=dpi)


---

### For context: what does the real lysozyme structure look like?

Two natural follow-up questions: could we look at the real lysozyme structure side by side with the generated one, and would the generated structure actually resemble it? Below, the **left panel** shows the real deposited structure, [1LYZ](https://www.rcsb.org/structure/1LYZ), and the **right panel** shows the final frame of the active-site-scaffolding design generated above, both rendered in the same viewer for direct comparison.

**Do not expect the right panel to look like the left one overall.** Motif scaffolding only copies in the ~10 fixed backbone atoms of the `A33-37`/`A50-54` windows around Glu35 and Asp52; every other residue — the rest of the fold, its overall size, its topology — is freely diffused from noise, with no reason at all to reproduce lysozyme's actual ~129-residue two-domain architecture. A generated structure that looks nothing like lysozyme overall is not a failure — it's exactly what motif scaffolding is designed to produce. The real test is below.


In [7]:
#@title Show the real lysozyme structure (1LYZ) for context {run: "auto"}
color = "chain"  #@param ["rainbow", "chain"]

lyz_pdb_str = pdb_to_string(get_pdb("1LYZ"))
gen_final_pdb_str = open(f"outputs/{path_motif}_0.pdb").read()

view = py3Dmol.view(viewergrid=(1, 2), js="https://3dmol.org/build/3Dmol.js")

# Left: the real deposited lysozyme structure, exactly as before.
view.addModel(lyz_pdb_str, "pdb", viewer=(0, 0))
view.setStyle({"cartoon": {"color": "spectrum"}}, viewer=(0, 0))
view.zoomTo(viewer=(0, 0))

# Right: the final frame of the active-site-scaffolding design generated above, for
# a direct side-by-side against the real structure on the left.
view.addModel(gen_final_pdb_str, "pdb", viewer=(0, 1))
if color == "rainbow":
    view.setStyle({"cartoon": {"color": "spectrum"}}, viewer=(0, 1))
else:  # "chain"
    for n, chain, c in zip(range(len(contigs_motif)), alphabet_list, pymol_color_list):
        view.setStyle({"chain": chain}, {"cartoon": {"color": c}}, viewer=(0, 1))
view.zoomTo(viewer=(0, 1))

view.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### The actual correctness check: does the *motif* geometry match?

The question that actually matters isn't "does the whole generated structure look like lysozyme" — it's "did RFdiffusion hold the fixed `A33-37`/`A50-54` windows in the correct relative 3D arrangement while designing a brand-new scaffold around them." We check this directly below, using one Kabsch fit computed from just the motif backbone atoms, shown two ways side by side:

- **Left panel:** only the motif fragments themselves, superposed directly. A tight overlap here — not the whole-structure view above — is the honest, quantitative test of whether motif scaffolding actually worked.
- **Right panel:** the exact same rigid rotation and translation (nothing re-fit — the identical transform from the left panel), applied to *every atom* of the whole generated structure, overlaid on the whole real lysozyme structure.

**Expect the right panel to look messy and tangled, not aligned.** Once the shared motif is forced to coincide, the rest of the freely diffused scaffold does not land anywhere near lysozyme's real two-domain fold — nothing outside the motif was ever constrained to resemble lysozyme, so nothing outside the motif does. That is the honest, direct illustration that the design is genuinely novel apart from the fixed catalytic site.


In [8]:
#@title Motif superposition: generated active site vs. real lysozyme {run: "auto"}
BACKBONE_ATOMS = ["N", "CA", "C"]

def _parse_pdb_backbone(pdb_str):
    """dict[(chain, resnum)][atom] -> (xyz, original_pdb_line); first model/altloc only."""
    coords = {}
    for line in pdb_str.splitlines():
        if not (line.startswith("ATOM") or line.startswith("HETATM")):
            continue
        atom_name = line[12:16].strip()
        if atom_name not in BACKBONE_ATOMS:
            continue
        chain = line[21].strip()
        resnum = int(line[22:26])
        xyz = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
        entry = coords.setdefault((chain, resnum), {})
        if atom_name not in entry:
            entry[atom_name] = (xyz, line)
    return coords


def _fixed_segments(contig):
    """Reproduce colabdesign.rf.utils.fix_pdb's own residue-renumbering logic, to map
    each fixed motif segment in a resolved contig string (e.g. "A33-37") to the
    residue range it ends up at in the generated structure's own numbering."""
    sub = [x.split("-") for x in contig.split("/")]
    L_init = 1
    segments = []
    for n, (a, b) in enumerate(sub):
        if a[0].isalpha():
            if n > 0:
                pa, pb = sub[n - 1]
                if pa[0].isalpha() and a[0] == pa[0]:
                    L_init += int(a[1:]) - int(pb) - 1
            L = int(b) - int(a[1:]) + 1
            segments.append({
                "ref_chain": a[0], "ref_start": int(a[1:]), "ref_end": int(b),
                "hal_start": L_init, "hal_end": L_init + L - 1,
            })
        else:
            L = int(b)
        L_init += L
    return segments


def kabsch_align(mobile, target):
    """Superpose `mobile` (N,3) onto `target` (N,3) with the optimal rotation (Kabsch
    algorithm, no reflection); returns (rotated_mobile, rmsd, R) -- the same approach
    already used to independently validate the motif RMSD reported in the Discussion
    below (computed from scratch, not relying on RFdiffusion's own internal log). `R`
    is exposed so the identical rigid transform can be reapplied below to every atom
    of the full generated structure, not just the motif backbone atoms used to fit it."""
    mobile_c = mobile - mobile.mean(axis=0)
    target_c = target - target.mean(axis=0)
    C = mobile_c.T @ target_c
    U, S, Vt = np.linalg.svd(C)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    D = np.diag([1, 1, d])
    R = Vt.T @ D @ U.T
    rotated = mobile_c @ R.T + target.mean(axis=0)
    rmsd = np.sqrt(((mobile_c @ R.T - target_c) ** 2).sum() / mobile.shape[0])
    return rotated, rmsd, R


def _rewrite_xyz(line, xyz):
    """Swap only the x/y/z columns of a real PDB ATOM/HETATM line, keeping everything
    else (serial, atom/residue name, chain, occupancy, element) intact and valid."""
    x, y, z = xyz
    return f"{line[:30]}{x:8.3f}{y:8.3f}{z:8.3f}{line[54:]}"


def _transform_all_atoms(pdb_str, R, mobile_mean, target_mean):
    """Apply the exact rigid-body transform Kabsch-fit on the motif backbone atoms
    above (rotation `R` about the motif's own centroid, then a shift to the real
    structure's motif centroid) to every ATOM/HETATM line of a full PDB string --
    moving the *whole* generated structure rigidly into lysozyme's reference frame,
    not just the ~10 residues the fit itself was computed from."""
    out_lines = []
    for line in pdb_str.splitlines():
        if line.startswith("ATOM") or line.startswith("HETATM"):
            xyz = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
            line = _rewrite_xyz(line, (xyz - mobile_mean) @ R.T + target_mean)
        out_lines.append(line)
    return "\n".join(out_lines)


# Recover which residues of the generated structure correspond to which residues of
# real 1LYZ straight from the resolved contig string `run_diffusion` already returned
# (contigs_motif) -- no need to re-parse RFdiffusion's own .trb metadata file.
segments = _fixed_segments(contigs_motif[0])

gen_pdb_str = open(f"outputs/{path_motif}_0.pdb").read()
ref_pdb_str = pdb_to_string(get_pdb("1LYZ"))
gen_coords = _parse_pdb_backbone(gen_pdb_str)
ref_coords = _parse_pdb_backbone(ref_pdb_str)

hal_residues, ref_residues = [], []
for seg in segments:
    for offset in range(seg["ref_end"] - seg["ref_start"] + 1):
        hal_residues.append(("A", seg["hal_start"] + offset))
        ref_residues.append((seg["ref_chain"], seg["ref_start"] + offset))

mobile = np.array([gen_coords[r][a][0] for r in hal_residues for a in BACKBONE_ATOMS])
target = np.array([ref_coords[r][a][0] for r in ref_residues for a in BACKBONE_ATOMS])
rotated, motif_rmsd, R = kabsch_align(mobile, target)

print(f"Motif backbone (N, CA, C) RMSD after Kabsch superposition: {motif_rmsd:.3f} Å")
print(f"  generated residues (this design's own numbering): {hal_residues}")
print(f"  real lysozyme residues (1LYZ, chain A):            {ref_residues}")

# Build a two-model overlay: the generated motif backbone (rotated into 1LYZ's own
# frame) plus the real 1LYZ motif backbone, so the geometric overlap is directly visible.
gen_lines, i = [], 0
for r in hal_residues:
    for a in BACKBONE_ATOMS:
        gen_lines.append(_rewrite_xyz(gen_coords[r][a][1], rotated[i]))
        i += 1
gen_lines.append("TER")

ref_lines = [ref_coords[r][a][1] for r in ref_residues for a in BACKBONE_ATOMS]
ref_lines.append("TER")

# Apply that exact same rigid transform (same R, same two centroids -- nothing
# re-fit) to every atom of the full generated structure, so the whole design lands
# in lysozyme's own reference frame rather than just its motif backbone atoms.
gen_full_aligned = _transform_all_atoms(gen_pdb_str, R, mobile.mean(axis=0), target.mean(axis=0))

view = py3Dmol.view(viewergrid=(1, 2), js="https://3dmol.org/build/3Dmol.js")

# Left: motif-only overlay -- the honest, quantitative correctness check.
view.addModel("\n".join(gen_lines), "pdb", viewer=(0, 0))
view.addModel("\n".join(ref_lines), "pdb", viewer=(0, 0))
view.setStyle({"model": 0}, {"stick": {"color": "orange"}}, viewer=(0, 0))
view.setStyle({"model": 1}, {"stick": {"color": "cyan"}}, viewer=(0, 0))
view.zoomTo(viewer=(0, 0))

# Right: the whole generated structure, rigidly moved by that same motif fit,
# overlaid on the whole real lysozyme structure. Expect this to look tangled, not
# aligned -- see the markdown above.
view.addModel(gen_full_aligned, "pdb", viewer=(0, 1))
view.addModel(ref_pdb_str, "pdb", viewer=(0, 1))
view.setStyle({"model": 0}, {"cartoon": {"color": "orange"}}, viewer=(0, 1))
view.setStyle({"model": 1}, {"cartoon": {"color": "cyan"}}, viewer=(0, 1))
view.zoomTo(viewer=(0, 1))

view.show()
print(f"orange = generated   |   cyan = real 1LYZ   |   motif-region RMSD = {motif_rmsd:.3f} Å")

# Sanity check: after applying the exact same rigid transform to the full generated
# structure, its motif-region backbone atoms should still land on the real motif
# atoms at (essentially) the same RMSD as above -- confirming this is the identical
# transform, not a separately (and possibly differently) fit one.
full_aligned_coords = _parse_pdb_backbone(gen_full_aligned)
full_aligned_motif = np.array([full_aligned_coords[r][a][0] for r in hal_residues for a in BACKBONE_ATOMS])
sanity_rmsd = np.sqrt(((full_aligned_motif - target) ** 2).sum() / target.shape[0])
print(f"Sanity check -- motif RMSD recomputed from the full-structure-aligned PDB: {sanity_rmsd:.3f} Å (should match the {motif_rmsd:.3f} Å above)")


Motif backbone (N, CA, C) RMSD after Kabsch superposition: 0.282 Å
  generated residues (this design's own numbering): [('A', 34), ('A', 35), ('A', 36), ('A', 37), ('A', 38), ('A', 50), ('A', 51), ('A', 52), ('A', 53), ('A', 54)]
  real lysozyme residues (1LYZ, chain A):            [('A', 33), ('A', 34), ('A', 35), ('A', 36), ('A', 37), ('A', 50), ('A', 51), ('A', 52), ('A', 53), ('A', 54)]


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

orange = generated   |   cyan = real 1LYZ   |   motif-region RMSD = 0.282 Å
Sanity check -- motif RMSD recomputed from the full-structure-aligned PDB: 0.282 Å (should match the 0.282 Å above)


---

## Discussion

- **Unconditional generation** shows RFdiffusion's raw generative prior: with no constraints at all, it still produces a plausible, well-packed fold — this is the same denoising machinery used underneath every other RFdiffusion task.
- **Motif scaffolding** is what makes this relevant to *enzyme* design specifically: by fixing only the catalytic residues (not the whole protein), RFdiffusion has to design a brand-new fold that nonetheless reproduces the precise 3D geometry the reaction chemistry actually needs. This is the same core idea used in real de novo enzyme design pipelines, just at a scale a live demo can afford.
- **Does the conditioning actually work?** We checked directly: holding the lysozyme Glu35/Asp52 motif fixed (`ActiveSite_ckpt.pt`) reproduces its true backbone geometry to ~0.3-0.8 Å RMSD across two independent random seeds, after best-fit alignment. An unconditioned control of matching length (`Base_ckpt.pt`, no motif constraint) lands at ~2.2 Å RMSD at the equivalent positions -- several times worse, as expected, since nothing forces unconstrained noise to reproduce a specific two-residue arrangement. That gap is the concrete evidence that the motif-conditioning mechanism is doing real work, not just producing plausible-looking structures by chance.
- Every output here is a **backbone only** — designed residues come out as glycine placeholders with no side chains, because RFdiffusion is not trained to output sequence. The usual next step (not run in this short demo, but included in the original [ColabDesign notebook](https://github.com/sokrypton/ColabDesign/blob/main/rf/examples/diffusion.ipynb) if you want to explore it) is **ProteinMPNN** to design an actual sequence for the new backbone, followed by **AlphaFold** to check that the sequence folds back into the intended structure.
- The exact contig window sizes above (`30-40`, `8-14`, the 5-residue fixed motifs) are a judgment call, not a uniquely correct answer — motif scaffolding is somewhat forgiving to reasonable window choices, but very small or very large windows can both hurt design success in practice.

---

<sub>**RFdiffusion** — Watson, J.L. et al. *De novo design of protein structure and function with RFdiffusion.* Nature 620, 1089–1100 (2023). Code & weights: [RosettaCommons/RFdiffusion](https://github.com/RosettaCommons/RFdiffusion) (BSD License, free for non-profit and for-profit use). Notebook adapted from Sergey Ovchinnikov's [ColabDesign](https://github.com/sokrypton/ColabDesign) (`rf/examples/diffusion.ipynb`), which this notebook's setup, `run_diffusion` plumbing, and 3D trajectory animation all directly build on. Weights mirrored at [soldatmat/CZAI_Summer_School-RFdiffusion_weights](https://huggingface.co/datasets/soldatmat/CZAI_Summer_School-RFdiffusion_weights).

**Hen egg-white lysozyme structure** — Diamond, R. *Real-space refinement of the structure of hen egg-white lysozyme.* J. Mol. Biol. 82, 371-391 (1974). PDB entry [1LYZ](https://www.rcsb.org/structure/1LYZ).</sub>
